# Model inference


In [3]:
!pip install wandb -q

In [4]:
from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

# Data Loading

In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

df = pd.read_csv('/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv')
df.columns = df.columns.str.strip()

train_df = df[df['Usage'] == 'Training'].reset_index(drop=True)
val_df = df[df['Usage'] == 'PublicTest'].reset_index(drop=True)
test_df = df[df['Usage'] == 'PrivateTest'].reset_index(drop=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Device:", device)

Train: 28709 Val: 3589 Test: 3589
Device: cuda


In [6]:
class FERDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        pixels = np.array(self.dataframe.iloc[idx]['pixels'].split(),
                          dtype=np.uint8).reshape(48, 48)
        image = Image.fromarray(pixels)
        if self.transform:
            image = self.transform(image)
        label = int(self.dataframe.iloc[idx]['emotion'])
        return image, label

augmented_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomCrop(48, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = FERDataset(train_df, transform=augmented_transform)
val_dataset = FERDataset(val_df, transform=val_transform)
test_dataset = FERDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Best Model Architecture

In [7]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = x + residual
        return torch.relu(x)


class BestModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm2d(64)
        self.res1 = ResidualBlock(64)
        self.res2 = ResidualBlock(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.res3 = ResidualBlock(128)
        self.res4 = ResidualBlock(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.res5 = ResidualBlock(256)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        self.pool = nn.MaxPool2d(2, 2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.4)
        self.fc = nn.Linear(512, 7)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.res1(x)
        x = self.res2(x)
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.res3(x)
        x = self.res4(x)
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))
        x = self.res5(x)
        x = torch.relu(self.bn4(self.conv4(x)))
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.drop(x)
        return self.fc(x)

# Training Best Model

In [8]:
model = BestModel().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

best_val_acc = 0
best_model_state = None

run = wandb.init(
    project="facial_expression_recognition",
    group="Inference",
    name="best_model_final",
    config={
        'architecture': 'DeepCNN_v4',
        'epochs': 40,
        'lr': 0.001,
        'weight_decay': 1e-4,
        'label_smoothing': 0.1,
        'scheduler': 'CosineAnnealing',
        'augmentation': True
    }
)

for epoch in range(40):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    wandb.log({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    print(f"Epoch {epoch+1}/40 | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

print(f"\nBest val accuracy: {best_val_acc:.4f}")
wandb.finish()

Epoch 1/40 | Train Acc: 0.3111 | Val Acc: 0.4012
Epoch 2/40 | Train Acc: 0.4718 | Val Acc: 0.4778
Epoch 3/40 | Train Acc: 0.5159 | Val Acc: 0.5157
Epoch 4/40 | Train Acc: 0.5425 | Val Acc: 0.5272
Epoch 5/40 | Train Acc: 0.5581 | Val Acc: 0.5269
Epoch 6/40 | Train Acc: 0.5687 | Val Acc: 0.5690
Epoch 7/40 | Train Acc: 0.5841 | Val Acc: 0.5770
Epoch 8/40 | Train Acc: 0.5920 | Val Acc: 0.5678
Epoch 9/40 | Train Acc: 0.5937 | Val Acc: 0.5874
Epoch 10/40 | Train Acc: 0.6053 | Val Acc: 0.5648
Epoch 11/40 | Train Acc: 0.6139 | Val Acc: 0.5896
Epoch 12/40 | Train Acc: 0.6213 | Val Acc: 0.6004
Epoch 13/40 | Train Acc: 0.6308 | Val Acc: 0.6160
Epoch 14/40 | Train Acc: 0.6357 | Val Acc: 0.6269
Epoch 15/40 | Train Acc: 0.6416 | Val Acc: 0.6197
Epoch 16/40 | Train Acc: 0.6467 | Val Acc: 0.6397
Epoch 17/40 | Train Acc: 0.6572 | Val Acc: 0.6258
Epoch 18/40 | Train Acc: 0.6655 | Val Acc: 0.6074
Epoch 19/40 | Train Acc: 0.6699 | Val Acc: 0.6397
Epoch 20/40 | Train Acc: 0.6796 | Val Acc: 0.6464
Epoch 21/

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_acc,▁▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████████
train_loss,█▆▅▅▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▃▄▄▄▅▅▅▆▅▆▆▆▇▆▇▇▆▇▇▇▇▇▇▇███████████████
val_loss,█▆▅▄▄▃▃▃▃▄▃▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,40
train_acc,0.80989
train_loss,0.86311
val_acc,0.6779
val_loss,1.17624


# Save Best Model as W&B Artifact

In [9]:
model.load_state_dict(best_model_state)
torch.save(best_model_state, '/kaggle/working/best_model_final.pth')
run = wandb.init(project="facial_expression_recognition", name="model_artifact")
artifact = wandb.Artifact('best_model_fer', type='model',
                          description='Best DeepCNN v4 model, val_acc=0.6799')
artifact.add_file('/kaggle/working/best_model_final.pth')
run.log_artifact(artifact)
wandb.finish()

In [10]:
class FERTestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        pixels = np.array(self.dataframe.iloc[idx]['pixels'].split(),
                          dtype=np.uint8).reshape(48, 48)
        image = Image.fromarray(pixels)
        if self.transform:
            image = self.transform(image)
        return image

test_csv = pd.read_csv('/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv')
print("Test set size:", len(test_csv))

test_dataset_final = FERTestDataset(test_csv, transform=val_transform)
test_loader_final = DataLoader(test_dataset_final, batch_size=64, shuffle=False)

model.eval()
all_preds = []

with torch.no_grad():
    for images in test_loader_final:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)

submission = pd.read_csv('/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/example_submission.csv')
print("Submission format:")
print(submission.head())

Test set size: 7178
Submission format:
   3
0  4
1  0
2  4
3  3
4  3


In [11]:
# create submission
submission_df = pd.DataFrame(all_preds, columns=['emotion'])
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print("Shape:", submission_df.shape)
print("Sample predictions:")
print(submission_df.head(10))
print("\nPrediction distribution:")
print(submission_df['emotion'].value_counts().sort_index())

Shape: (7178, 1)
Sample predictions:
   emotion
0        0
1        1
2        0
3        6
4        3
5        3
6        4
7        6
8        4
9        2

Prediction distribution:
emotion
0    1014
1      90
2     924
3    1748
4    1195
5     825
6    1382
Name: count, dtype: int64


In [12]:
run = wandb.init(project="facial_expression_recognition", name="submission_artifact")
artifact = wandb.Artifact('submission_fer', type='dataset',
                          description='Test predictions from best DeepCNN v4 model')
artifact.add_file('/kaggle/working/submission.csv')
run.log_artifact(artifact)
wandb.finish()